<a href="https://colab.research.google.com/github/Mohammed-Atef2004/Doc2Pod-Services/blob/main/Copy_of_Doc2Pod_With_TTS_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title ⚙️ Step 0: Logging Setup
# @markdown Run this cell first to initialize the unified logging system

import logging
import sys
import os

def setup_system_logger(log_filename="doc2pod_api.log"):
    """Initializes a unified logger for both console and file output."""
    for handler in logging.root.handlers[:]:
        logging.root.removeHandler(handler)

    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S"
    )

    file_handler = logging.FileHandler(log_filename, mode="a", encoding="utf-8")
    file_handler.setFormatter(formatter)

    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setFormatter(formatter)

    log = logging.getLogger()
    log.setLevel(logging.INFO)
    log.addHandler(file_handler)
    log.addHandler(console_handler)
    return log

logger = setup_system_logger()
logger.info("🚀 Logging System Initialized successfully!")

2026-06-26 11:23:27 | INFO     | root | 🚀 Logging System Initialized successfully!


In [ ]:
# @title 🚑 Emergency Fix: Force NumPy & Pandas Compatibility
# @markdown Run this cell immediately after "Restart Session".
# @markdown It reinstalls both NumPy and Pandas to ensure they match.

import os

logger.info("🔄 Fixing dependencies... (This takes ~30 seconds)")

# 1. Uninstall existing conflicting versions
os.system("pip uninstall -y numpy pandas")

# 2. Install compatible versions
# We force NumPy < 2.0 and let pip find a Pandas version that matches it.
os.system("pip install \"numpy<2.0\" pandas")

# 3. Verify
import numpy
import pandas
logger.info(f"✅ NumPy Version: {numpy.__version__}")
logger.info(f"✅ Pandas Version: {pandas.__version__}")

if numpy.__version__.startswith("2"):
    logger.error("❌ ERROR: NumPy is still 2.x. Please Restart Session and try again.")
else:
    logger.info("👉 Success! Now run the rest of the notebook.")

2026-06-26 11:23:31 | INFO     | root | 🔄 Fixing dependencies... (This takes ~30 seconds)


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
# @title 🛠️ Step 1: Install Dependencies & System Tools
# @markdown This installs OCR tools (PaddlePaddle), Vector Database (Chroma), and AI models.
# @markdown **Note:** This installs specific nightly builds of PyTorch. It may take 3-5 minutes.

# System Tools (Required for PDF conversion)
!apt-get update -qq
!apt-get install -y poppler-utils -qq
!which pdfinfo  # Verify installation

# PaddleOCR (for extracting text/tables)
!pip install paddlepaddle-gpu==3.2.1 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/ --trusted-host www.paddlepaddle.org.cn -q
!pip install paddlex==3.3.12 "paddleocr[all]" --trusted-host pypi.org --trusted-host files.pythonhosted.org -q
# Utils (PDF handling, Regex, etc.)
!pip install langchain==0.0.354 pdf2image sentence-transformers regex uuid beautifulsoup4 pylatexenc -q

# PyTorch (Nightly Build )
!pip install torch==2.11.0.dev20260107+cu126 torchvision==0.25.0.dev20260107+cu126 torchaudio==2.11.0.dev20260107+cu126 --index-url https://download.pytorch.org/whl/nightly/cu126 -q
!pip install transformers accelerate --upgrade -q

# Vision Language Model Utils
!pip install qwen-vl-utils -q

# Vector Database & Data Handling
!pip install chromadb pandas -q

# --- Unsloth High-Performance Engine ---
!pip install unsloth

# Numpy Fix (Uninstall/Reinstall to prevent version conflicts)
!pip uninstall -y numpy -q
!pip install numpy==1.26.4 -q

# --- Server & Tunnel Dependencies ---

# FastAPI
!pip install fastapi uvicorn requests python-multipart -q

# Cloudflare Tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

!apt install ffmpeg -y
!pip install supabase

logger.info("✅ Step 1 Complete: All dependencies installed.")

In [ ]:
import os
# 1. Clear out the old versions entirely
!pip uninstall -y torch torchvision torchaudio torchcodec torchao -q

# 2. Install the LATEST nightly builds (which satisfy the >= 2.11.0 requirement)
!pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu126 -q
!pip install --pre torchcodec --index-url https://download.pytorch.org/whl/nightly/cu126 -q

print("\n✅ Latest Nightly installed. Please click 'RESTART SESSION' below, then run Step 0 and Step 2.")

In [ ]:
# @title 📚 Step 2: Import Libraries
# @markdown Loads standard tools, models, and sets up the unified directory structure.

import gc
import re
import uuid
import html
import json
import glob
import time
import ctypes
import shutil
import tempfile
import threading
import subprocess
from pathlib import Path
from typing import Any, List, Dict
from io import StringIO

import torch
import paddle
import psutil
import chromadb
import numpy as np
import pandas as pd
import soundfile as sf
from PIL import Image
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm
from google.colab import drive
from google.colab import userdata

# AI & Vision
from qwen_vl_utils import process_vision_info
from paddleocr import PPStructureV3, PaddleOCRVL
from pylatexenc.latex2text import LatexNodes2Text
from sentence_transformers import SentenceTransformer
from pdf2image import convert_from_path, pdfinfo_from_path
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor, AutoModelForCausalLM, AutoTokenizer
from sklearn.metrics.pairwise import cosine_similarity
from huggingface_hub import snapshot_download

# Server
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel
import requests
from supabase import create_client
import uvicorn
import nest_asyncio
import asyncio

logger.info("✅ Step 2 Complete: Libraries imported successfully.")

In [ ]:
# @title 📂 Step 3: Mount Drive & Configure Unified Workspace
# @markdown We set up a comprehensive directory structure to handle the full pipeline:
# @markdown PDF -> Images -> OCR JSON -> Embeddings -> ChromaDB -> Podcast Audio.

import os
from pathlib import Path
from google.colab import drive

# Mount Drive
drive.mount('/content/drive')

# Define Root Path
BASE_DIR = Path("/content/drive/MyDrive/Doc2Pod_System")


# Define Sub-Directories
DATA_DIR = BASE_DIR / "Data"
PDF_DIR = DATA_DIR / "pdfs"                    # Raw PDFs go here
IMAGE_DIR = DATA_DIR / "images"                # Extracted images from slides
JSON_DIR = DATA_DIR / "json"                   # Raw OCR output (per page)
UNIFIED_DIR = DATA_DIR / "unified_json"        # Cleaned/Merged text chunks
EMBEDDING_DIR = DATA_DIR / "Embedding_inputs"  # Pre-processed data for Vector DB
DB_DIR = DATA_DIR / "chroma_db"                # The Vector Database (Chroma)
MODE1_DIR = BASE_DIR / "Mode1"
MODE2_DIR = BASE_DIR / "Mode2"
MODE3_DIR = BASE_DIR / "Mode3"

# 4. Create All Directories
folders_to_create = [
    BASE_DIR, DATA_DIR, PDF_DIR, IMAGE_DIR, JSON_DIR,
    UNIFIED_DIR, EMBEDDING_DIR, DB_DIR, MODE1_DIR, MODE2_DIR, MODE3_DIR
]

for d in folders_to_create:
    d.mkdir(parents=True, exist_ok=True)

# 5. Print Summary
logger.info(f"✅ Workspace Ready at: {BASE_DIR}")
logger.info(f"   📂 Data Folder:     {DATA_DIR.name}")
logger.info(f"      ├── Input PDFs:  {PDF_DIR.name}")
logger.info(f"      ├── Vector DB:   {DB_DIR.name}")
logger.info(f"      └── Images:      {IMAGE_DIR.name}")
logger.info(f"      Output Modes:")
logger.info(f"      ├── Mode 1:      {MODE1_DIR.name}")
logger.info(f"      ├── Mode 2:      {MODE2_DIR.name}")
logger.info(f"      └── Mode 3:      {MODE3_DIR.name}")

logger.info("\n✅ Step 3 Complete: Drive mounted & Directories structured.")

In [ ]:
# @title 🧠 Step 4: Model Managers (Lazy Loading) & Memory Sweeper

def force_system_ram_cleanup():
    gc.collect()
    torch.cuda.empty_cache()
    try:
        libc = ctypes.CDLL("libc.so.6")
        libc.malloc_trim(0)
    except Exception:
        pass

def load_PaddleOCRVL():
    device = "gpu" if torch.cuda.is_available() else "cpu"
    logger.info(f"[*] Loading PaddleOCR on {device}...")
    pipeline = PaddleOCRVL(device=device, use_layout_detection=True, format_block_content=True, use_chart_recognition=True)
    return pipeline

def load_qwen3_vl_model():
    logger.info("[*] Loading Qwen3-VL (Vision Model)...")
    model = Qwen3VLForConditionalGeneration.from_pretrained("Qwen/Qwen3-VL-4B-Instruct", torch_dtype=torch.float16, device_map="auto", trust_remote_code=True)
    processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-4B-Instruct", min_pixels=256*28*28, max_pixels=768*28*28, trust_remote_code=True)
    model.eval()
    return model, processor

def load_qwen3_instruct_model():
    logger.info("[*] Loading Qwen-Text Model (Standard Transformers with Low Memory)...")
    model_id = "Fatma04/Arabic-Podcast-Qwen-16bit"

    tokenizer = AutoTokenizer.from_pretrained(model_id)

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="auto",
        trust_remote_code=True,
        low_cpu_mem_usage=True
    )
    model.eval()
    return model, tokenizer

class LazyPipelineManager:
    def __init__(self):
        self._vl_pipeline = None
        self._vl_loaded = False
        self._qwen3_vl = None
        self._processor = None
        self._qwen3_vl_loaded = False
        self._qwen3_instruct = None
        self._instruct_tokenizer = None
        self._instruct_loaded = False
        self._voice_model_ready = False

    def get_vl_pipeline(self):
        if not self._vl_loaded:
            self._vl_pipeline = load_PaddleOCRVL()
            self._vl_loaded = True
        return self._vl_pipeline

    def get_qwen3_vl_model(self):
        if not self._qwen3_vl_loaded:
            self._qwen3_vl, self._processor = load_qwen3_vl_model()
            self._qwen3_vl_loaded = True
        return self._qwen3_vl, self._processor

    def get_qwen3_instruct_model(self):
        if not self._instruct_loaded:
            self._qwen3_instruct, self._instruct_tokenizer = load_qwen3_instruct_model()
            self._instruct_loaded = True
        return self._qwen3_instruct, self._instruct_tokenizer


    def ensure_voice_model_loaded(self):

        if self._voice_model_ready:
            return

        logger.info("🎤 Initializing Voice Model (First Time Setup)...")

        if not os.path.exists("/content/VibeVoice"):
            subprocess.run(["git", "clone", "https://github.com/vibevoice-community/VibeVoice.git", "/content/VibeVoice"])

        os.system("pip install -q -e /content/VibeVoice")
        os.system("pip install -q accelerate huggingface_hub soundfile ipython")

        REPO_ID = "MohammedEhab20/vibe-voice-egyptian-cfg35"
        MODEL_DIR = "/content/egyptian-model"

        logger.info("⬇️ Downloading TTS model from Hugging Face...")
        snapshot_download(repo_id=REPO_ID, local_dir=MODEL_DIR)

        voice_dest = "/content/VibeVoice/demo/voices"
        os.makedirs(voice_dest, exist_ok=True)

        logger.info("🎤 Setting up reference voices...")
        shutil.copy(f"{MODEL_DIR}/voices/egyptian_female.wav", f"{voice_dest}/egyptian_female.wav")
        shutil.copy(f"{MODEL_DIR}/voices/egyptian_male.wav", f"{voice_dest}/egyptian_male.wav")

        logger.info("✅ Voice Model configuration complete!")
        self._voice_model_ready = True


    def unload_all(self):
        logger.info("[*] Unloading all models from GPU...")
        if self._vl_pipeline:
            del self._vl_pipeline
            self._vl_pipeline = None
            self._vl_loaded = False

        if self._qwen3_vl:
            del self._qwen3_vl
            self._qwen3_vl = None
            del self._processor
            self._processor = None
            self._qwen3_vl_loaded = False

        if self._qwen3_instruct:
            del self._qwen3_instruct
            self._qwen3_instruct = None
            del self._instruct_tokenizer
            self._instruct_tokenizer = None
            self._instruct_loaded = False

        force_system_ram_cleanup()
        logger.info("[*] GPU & System RAM cleared completely.")

logger.info("✅ Step 4 Complete: Model Managers defined.")

In [ ]:
# @title 🧹 Step 5: Text Cleaning & Normalization Utils
# @markdown Helper functions to clean code snippets, convert HTML tables to text, and fix LaTeX formatting.

# Constants for LaTeX normalization
SUPERSCRIPTS = {"0": "⁰", "1": "¹", "2": "²", "3": "³", "4": "⁴", "5": "⁵", "6": "⁶", "7": "⁷", "8": "⁸", "9": "⁹", "+": "⁺", "-": "⁻", "=": "⁼", "T": "ᵀ", "n": "ⁿ", "i": "ⁱ"}
SUBSCRIPTS = {"0": "₀", "1": "₁", "2": "₂", "3": "₃", "4": "₄", "5": "₅", "6": "₆", "7": "₇", "8": "₈", "9": "₉", "i": "ᵢ", "j": "ⱼ", "k": "ₖ", "n": "ₙ"}

def normalize_code(raw_code):
    """Cleans up OCR errors common in code blocks (e.g. '0' vs 'O')."""

    if not raw_code.strip():
       return "Code snippet: [empty]"

    lines = raw_code.split('\n')
    cleaned_lines = []

    for line in lines:
        stripped = line.rstrip()
        stripped = re.sub(r'\bO\b', '0', stripped)
        stripped = re.sub(r'\b[lI]\b', '1', stripped)
        stripped = re.sub(r'\bS\b(?=\d)', '5', stripped)
        stripped = re.sub(r'! =', '!=', stripped)
        stripped = re.sub(r'< =', '<=', stripped)
        stripped = re.sub(r'> =', '>=', stripped)
        cleaned_lines.append(stripped)

    normalized = '\n'.join(cleaned_lines).strip()

    if not normalized:
       return "Code snippet: [unrecognized content]"

    return f"Code snippet:\n{normalized}"


def html_table_to_text(html_table):
    """Parses HTML tables returned by PaddleOCR into pipe-separated text."""

    soup = BeautifulSoup(html_table, "html.parser")
    table = soup.find("table")

    if not table:
       return "Table"

    grid, rowspan_map = [], {}
    for tr in table.find_all("tr"):
        row, col_idx = [], 0
        while col_idx in rowspan_map:
            row.append(rowspan_map[col_idx]["text"])
            rowspan_map[col_idx]["rows"] -= 1
            if rowspan_map[col_idx]["rows"] == 0:
               del rowspan_map[col_idx]
            col_idx += 1
        for cell in tr.find_all(["td", "th"], recursive=False):
            text = cell.get_text(" ", strip=True) or "—"
            text = latex_to_unicode(text)
            colspan = int(cell.get("colspan", 1))
            rowspan = int(cell.get("rowspan", 1))
            for _ in range(colspan):
                row.append(text)
                if rowspan > 1:
                   rowspan_map[col_idx] = {"text": text, "rows": rowspan - 1}
                col_idx += 1
        grid.append(row)
    max_cols = max(len(r) for r in grid)
    for r in grid: r.extend([""] * (max_cols - len(r)))
    lines = []
    for row in grid:
        if any(cell.strip() for cell in row):
           lines.append(" | ".join(row))
    return "\n".join(lines)

def normalize_latex(text):
    """Basic cleanup for LaTeX strings."""

    text = re.sub(r"\${1,2}", "", text)
    text = text.replace("\\\\", "\\")
    text = re.sub(r"\\quad|\\qquad|\\,", " ", text)
    text = re.sub(r"\\begin\{.*?\}|\\end\{.*?\}", "", text)
    text = re.sub(r"\b[clr]{1,3}\b", "", text)
    return text

def latex_to_unicode(text):
    """Converts LaTeX math symbols to their Unicode equivalents for better readability."""
    if not text:
      return text
    text = normalize_latex(text)
    try:
       text = LatexNodes2Text().latex_to_text(text)
    except Exception:
       pass

    replacements = {
        r"\\theta": "θ", r"\\alpha": "α", r"\\beta": "β", r"\\gamma": "γ",
        r"\\lambda": "λ", r"\\mu": "μ", r"\\sigma": "σ", r"\\pi": "π",
        r"\\geq": "≥", r"\\leq": "≤", r"\\neq": "≠", r"\\approx": "≈",
        r"\\pm": "±", r"\\cdot": "·", r"\\times": "×", r"\\rightarrow": "→", r"\\infty": "∞"
    }
    for k, v in replacements.items():
        text = re.sub(k, v, text)

    # Handle superscripts/subscripts
    text = re.sub(r"\^\{([^}]+)\}",lambda m: "".join(SUPERSCRIPTS.get(c, c) for c in m.group(1)),text)
    text = re.sub(r"\^\(([^)]+)\)",lambda m: "".join(SUPERSCRIPTS.get(c, c) for c in m.group(1)),text)
    text = re.sub(r"_\{([^}]+)\}",lambda m: "".join(SUBSCRIPTS.get(c, c) for c in m.group(1)),text)
    text = re.sub(r"\^([0-9Tni+\-=])",lambda m: SUPERSCRIPTS.get(m.group(1), m.group(1)),text)
    text = re.sub(r"_([0-9ijkn])",lambda m: SUBSCRIPTS.get(m.group(1), m.group(1)),text)
    text = re.sub(r"\\[a-zA-Z]+", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def is_personal_info(text):
    """Detect admin/link-only content to exclude from chunks."""
    if not text.strip():
        return False

    lower_text = text.lower()

    # ❌ Admin content
    admin_keywords = [
        "office hours", "grading scheme", "grading policy",
        "course policy", "course outline", "helpful resources",
        "midterm exam", "final exam", "syllabus"
    ]
    if any(kw in lower_text for kw in admin_keywords):
        return True

    # ❌ Link-only content (e.g., YouTube links)
    url_patterns = [r'https?://', r'www\.', r'\.com', r'\.eg']
    if any(re.search(pattern, text) for pattern in url_patterns):
        # If the text is mostly a URL (short + contains link)
        words = text.split()
        if len(words) <= 3 and any(re.search(p, text) for p in url_patterns):
            return True

    # ❌ Email/phone patterns
    email_pattern = r"[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+"
    phone_pattern = r"\+?\d[\d\s-]{5,}\d"
    office_pattern = r"\b(Room|Building|Office)\b"
    if (re.search(email_pattern, text) or
        re.search(phone_pattern, text) or
        re.search(office_pattern, text)):
        return True

    return False

logger.info("✅ Step 5 Complete: Text utilities loaded.")

In [ ]:
# @title 👁️ Step 6: Visual Understanding (Qwen-VL)
# @markdown Logic to recover text from charts and diagrams using the Vision Language Model.

def recover_image_block_qwen(bbox, page_img, pipeline_manager, max_res=800):
    """
    Crops an image region and sends it to Qwen-VL for description.
    """
    x1, y1, x2, y2 = map(int, bbox)
    crop_w, crop_h = x2-x1, y2-y1

    # Skip tiny noise artifacts
    if crop_w < 50 or crop_h < 50:
        return {"type": "image", "text": "[Tiny image skipped]", "recovered": False}

    # Resize if too large to save token cost/time
    cropped = page_img.crop((x1, y1, x2, y2))
    scale = min(max_res / crop_w, max_res / crop_h, 1.0)
    if scale < 1.0:
        cropped = cropped.resize((int(crop_w*scale), int(crop_h*scale)), Image.Resampling.LANCZOS)

    if pipeline_manager:
        model, processor = pipeline_manager.get_qwen3_vl_model()
        try:
            # Re-crop from original for maximum quality
            cropped = page_img.crop((x1, y1, x2, y2))

            messages = [{
                "role": "user",
                "content": [
                    {"type": "image", "image": cropped},
                    {"type": "text", "text": (
                        "Extract and explain all readable textual and semantic information from this image. "
                        "If it is a chart, diagram, table, or equation, describe it clearly in technical English."
                        "If absolutely nothing is readable, output exactly: [NO READABLE CONTENT]"
                    )}
                ]
            }]

            text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            image_inputs, _ = process_vision_info(messages)

            inputs = processor(
                text=[text_prompt],
                images=image_inputs,
                return_tensors="pt"
            ).to(model.device)

            # Generate description
            with torch.inference_mode():
                generated_ids = model.generate(**inputs, max_new_tokens=512)

            output_text = processor.batch_decode(
                generated_ids[:, inputs.input_ids.shape[1]:],
                skip_special_tokens=True
            )[0].strip()

            # Cleanup
            del inputs, generated_ids, image_inputs, text_prompt, messages
            cropped.close()
            torch.cuda.empty_cache()
            gc.collect()

            if "NO READABLE CONTENT" in output_text.upper():
              return {"text": "[No content extracted]", "type": "image", "recovered": False}

            return {
                "text": output_text if output_text else "[No content extracted]",
                "type": "image",
                "recovered": bool(output_text)
            }

        except Exception as e:
            try:
                cropped.close()
                del cropped
                torch.cuda.empty_cache()
            except: pass
            logger.error(f"❌ Qwen-VL image recovery failed: {e}")
            return {"text": f"[Image recovery failed: {str(e)}]", "type": "image", "recovered": False}

logger.info("✅ Step 6 Complete: Vision Logic loaded.")

In [ ]:
# @title 📄 Step 7: Layout Analysis Helper
# @markdown Helper function to determine if two text blocks should be merged based on distance and alignment.

def can_merge(prev_unit, blk, page_no, y_ratio=0.1, indent_ratio=0.05):
    """
    Checks if 'blk' is visually close enough to 'prev_unit' to be considered the same paragraph.
    """
    if prev_unit is None:
       return False
    if prev_unit["page"] != page_no:
       return False

    curr_type = blk.get("label", "text")
    prev_type = prev_unit["type"]

    # Only merge same-type blocks (text with text)
    if curr_type != prev_type:
      return False
    if curr_type not in {"text", "paragraph"}:
      return False

    # Check geometric proximity
    prev_x1, prev_y1, prev_x2, prev_y2 = prev_unit["bbox"]
    curr_x1, curr_y1, curr_x2, curr_y2 = blk["bbox"]

    prev_height = prev_y2 - prev_y1
    page_width = max(prev_x2, curr_x2)

    # Check Vertical Gap
    max_gap = max(prev_height * y_ratio, 10)
    if abs(curr_y1 - prev_y2) > max_gap:
      return False

    # Check Indentation difference
    max_indent_diff = max(page_width * indent_ratio, 10)
    if abs(curr_x1 - prev_x1) > max_indent_diff:
       return False

    # Heuristic: Don't merge if previous line ends with sentence terminator
    prev_text = prev_unit["text"].strip()
    if prev_text.endswith((".", ":", "؟", "!", ";")):
       return False

    # Heuristic: Don't merge bullets
    bullet_pattern = r"^(\-|\*|\u2022|\d+\.)\s+"
    if re.match(bullet_pattern, blk.get("content", "").strip()):
       return False

    # Heuristic: Don't merge headers
    if prev_text.isupper() and len(prev_text.split()) < 6:
       return False

    return True

logger.info("✅ Step 7 Complete: Layout helper loaded.")

In [ ]:
# @title  Step 8: Raw Block Processing Logic
# @markdown Functions to convert raw OCR blocks into logical units (Paragraphs, Tables, Image Descriptions).

def to_dict(obj: Any) -> Any:
    """Helper to make custom objects JSON serializable."""
    if obj is None: return None
    if hasattr(obj, 'numpy') and callable(obj.numpy):
        try: return obj.numpy().tolist()
        except: return str(obj)
    elif isinstance(obj, np.ndarray): return obj.tolist()
    elif isinstance(obj, Path): return str(obj)
    elif isinstance(obj, (bytes, bytearray)): return obj.decode('utf-8', errors='replace')
    elif isinstance(obj, (tuple, set)): return [to_dict(item) for item in obj]
    elif hasattr(obj, '__dict__') and not isinstance(obj, type): return to_dict(obj.__dict__)
    elif isinstance(obj, dict): return {k: to_dict(v) for k, v in obj.items()}
    elif isinstance(obj, list): return [to_dict(item) for item in obj]
    try:
        json.dumps(obj)
        return obj
    except: return str(obj)

    return True


def is_valid_title(text):
    """Heuristic to check if a text block looks like a Section Header."""
    if not isinstance(text, str):
        return False
    text = text.strip()
    if len(text) < 5 or len(text) > 100:  # ← Increased min length
        return False

    # Reject LaTeX/math formulas as titles
    latex_patterns = [r'\$.*?\$', r'\\[a-zA-Z]+', r'\{.*?\}']
    if any(re.search(p, text) for p in latex_patterns):
        return False

    # Reject bullets/lists
    if text.startswith(('•', '-', '—', '*', '●', '·', '→')):
        return False

    # Reject administrative content
    admin_keywords = [
        "office hours", "grading scheme", "grading policy",
        "course policy", "course outline", "helpful resources",
        "midterm exam", "final exam", "syllabus"
    ]
    if any(kw in text.lower() for kw in admin_keywords):
        return False

    # ✅ Accept titles with:
    # - Multiple words (≥2)
    # - No trailing punctuation
    # - Not sentence-like
    words = text.split()
    if len(words) < 2:
        return False
    if text.endswith(('.', '!', ';')) and not text.endswith('...'):
        return False
    structural_keywords = ["comparison", "levels", "deployment", "challenges", "protocols", "domains"]
    if any(kw in text.lower() for kw in structural_keywords):
        return True

    return True

def blocks_to_units(pdf_data, pipeline_manager):
    """
    Iterates through OCR blocks, merges neighbors, parses tables, and recovers images using Qwen-VL.
    """
    units = []
    unit_idx = 0

    for page in pdf_data["pages"]:
        page_no = page["page_number"]
        page_img = Image.open(page["image_path"])
        blocks = page.get("parsing_res_list", [])

        # Filter and Sort Blocks
        valid_blocks = [b for b in blocks if b.get("bbox") and len(b.get("bbox", [])) == 4]
        blocks = sorted(valid_blocks, key=lambda b: (b.get("bbox")[1], b.get("bbox")[0]))

        prev_unit = None
        for blk in blocks:
            text = blk.get("content", "").strip()
            if is_personal_info(text): continue

            t = blk.get("label", "text")
            # Skip noise
            if t == "number" or (t == "text" and len(text) <= 2): continue

            # Handle Tables
            if t in ("table", "chart"):
                text = html_table_to_text(text)
                text = html.unescape(text)

            # Handle Images (The Vision Part)
            if t in ("table","image", "chart") and ((not text) or len(text) <= 2 or (text=="Table")):
                bbox = blk.get("bbox", [0,0,0,0])
                recovered = recover_image_block_qwen(bbox, page_img, pipeline_manager)

                if recovered.get("recovered"):
                    units.append({
                        "text": recovered["text"],
                        "type": "image",
                        "page": page_no,
                        "unit_id": f"u{unit_idx:06d}",
                        "bbox": bbox
                    })
                    unit_idx += 1
                    continue
                else:
                    continue

            if not text: continue

            # Attempt to Merge with previous block (for broken paragraphs)
            if prev_unit and can_merge(prev_unit, blk, page_no):
                prev_unit["text"] += " " + text
                prev_unit["bbox"][2] = max(prev_unit["bbox"][2], blk["bbox"][2])
                prev_unit["bbox"][3] = max(prev_unit["bbox"][3], blk["bbox"][3])
                continue

            # New Unit
            units.append({
                "text": text,
                "type": t,
                "page": page_no,
                "unit_id": f"u{unit_idx:06d}",
                "bbox": blk.get("bbox", [0,0,0,0])
            })
            prev_unit = units[-1]
            unit_idx += 1

        page_img.close()
    return units

logger.info("✅ Step 8 Complete: Block processors defined.")

In [ ]:
# @title 🍕 Step 9: Semantic Chunking & Embedding Prep
# @markdown Groups text by headers, excludes admin pages, and enforces page-aligned subchunking for coherence.

def extract_intro_chunks(units, first_title_idx):
  """Extracts non-admin introductory units before the first main title."""

  if first_title_idx is None or first_title_idx <= 0:
        return []

  intro_units = [
      u for u in units[:first_title_idx]
      if u.get("text", "").strip() and not is_personal_info(u.get("text", ""))
  ]

  if intro_units:
      return [{
          "concept": "Introduction",
          "content": intro_units,
          "page_start": intro_units[0]["page"]
      }]
  return []


def process_units_to_chunks(units, emb_model):
    """
    Hybrid chunking strategy that dynamically adapts to the document's layout.
    """
    LARGE_ELEMENT_TYPES = {"image", "table", "chart", "display_formula", "algorithm"}
    level_pattern = r'\bLevel[-\s]*\d+\b'

    # =========================
    # 1. GROUP BY PAGE & FILTER ADMIN
    # =========================
    page_to_units = {}
    admin_pages = set()

    for u in units:
        page_no = u["page"]
        page_to_units.setdefault(page_no, []).append(u)

    for page_no, page_units in page_to_units.items():
        if page_units:
            first_text = page_units[0].get("text", "").strip()
            if is_personal_info(first_text):
                logger.info(f"Skipping entire page {page_no} (admin title): '{first_text}'")
                admin_pages.add(page_no)

    valid_units = [u for u in units if u["page"] not in admin_pages]
    valid_page_to_units = {}
    for u in valid_units:
        valid_page_to_units.setdefault(u["page"], []).append(u)

    # =========================
    # 2. DETECT CANDIDATE TITLES
    # =========================
    strategy_1_pages = set()
    strategy_2_pages = set()

    for page_no, page_units in valid_page_to_units.items():
        if not page_units: continue

        for unit in page_units[:3]:
            text = unit.get("text", "").strip()
            y1 = unit.get("bbox", [0, 0, 0, 0])[1]

            is_ocr_title = unit.get("type") == "paragraph_title"
            is_positional_title = (y1 < 300) and (5 < len(text) < 100) and (unit.get("type") not in LARGE_ELEMENT_TYPES)
            is_level_title = bool(re.search(level_pattern, text, re.IGNORECASE)) and unit.get("type") not in LARGE_ELEMENT_TYPES
            is_valid = is_valid_title(text)

            if (is_ocr_title or is_positional_title or is_level_title) and is_valid:
                content_weight = sum(5 if u.get("type") in LARGE_ELEMENT_TYPES else 1 for u in page_units if u != unit)
                structural_keywords = ["comparison", "levels", "level", "deployment", "challenges", "protocols", "domains"]
                has_structural_keyword = any(kw in text.lower() for kw in structural_keywords) and unit.get("type") not in LARGE_ELEMENT_TYPES

                if len(page_units) == 1 or content_weight < 1 or has_structural_keyword or is_level_title:
                    strategy_1_pages.add(page_no)
                    logger.info(f"✅ [STRATEGY 1] Title detected on Page {page_no}: '{text}'")
                else:
                    strategy_2_pages.add(page_no)
                    logger.info(f"✅ [STRATEGY 2] Title detected on Page {page_no}: '{text}'")

                break
    # =========================
    # 3. FILTER TOP LEVEL SECTIONS
    # =========================
    real_strategy_1_pages = {p for p in strategy_1_pages if p != 1}

    if len(real_strategy_1_pages) >= 3:
        main_title_pages = real_strategy_1_pages
        logger.info(f"✅ Using {len(real_strategy_1_pages)} Strategy 1 pages as top-level sections.")
    else:
        main_title_pages = strategy_1_pages | strategy_2_pages
        logger.info(f"⚠️ Only {len(real_strategy_1_pages)} Strategy 1 pages found. Keeping ALL valid titles (Strategy 1 + Strategy 2).")

    if main_title_pages:
        logger.info(f"Final title pages used: {sorted(list(main_title_pages))}")

    # =========================
    # 4. BUILD INITIAL CHUNKS
    # =========================
    first_main_title_idx = next(
        (i for i, u in enumerate(valid_units)
         if u["page"] in main_title_pages and
         (is_valid_title(u.get("text", "").strip()) or re.search(level_pattern, u.get("text", ""), re.IGNORECASE)) and u.get("type") not in LARGE_ELEMENT_TYPES),
        None
    )

    intro_chunks = extract_intro_chunks(valid_units, first_main_title_idx)
    if first_main_title_idx is not None and first_main_title_idx > 0:
        valid_units = valid_units[first_main_title_idx:]

    initial_chunks = []
    current_chunk = {"concept": "General Section", "content": [], "page_start": None}
    used_title_pages = set()

    for u in valid_units:
        u_text = u.get("text", "").strip()
        if not u_text or is_personal_info(u_text): continue

        page_no = u["page"]
        y1 = u.get("bbox", [0, 0, 0, 0])[1]
        is_level_title = bool(re.search(level_pattern, text, re.IGNORECASE)) and u.get("type") not in LARGE_ELEMENT_TYPES
        is_eligible_title = (
            page_no in main_title_pages
            and page_no not in used_title_pages
            and (u.get("type") == "paragraph_title" or (y1 < 300 and len(u_text) < 100 and u.get("type") not in LARGE_ELEMENT_TYPES) or is_level_title)
            and (is_valid_title(u_text) or is_level_title)
        )

        if is_eligible_title:
            if current_chunk["content"]:
                initial_chunks.append(current_chunk)
            current_chunk = {"concept": u_text, "content": [], "page_start": page_no}
            used_title_pages.add(page_no)
            logger.info(f"Starting new chunk: '{u_text}' (Page {page_no})")
        else:
            if current_chunk["page_start"] is None:
                current_chunk["page_start"] = page_no
            current_chunk["content"].append(u)

    if current_chunk["content"]:
        initial_chunks.append(current_chunk)

    logger.info(f"Initial chunk count: {len(initial_chunks)}")

    # =========================
    # 5. MERGE SIMILAR TITLES
    # =========================
    final_chunks = []
    i = 0
    while i < len(initial_chunks):
        curr_chunk = initial_chunks[i]
        j = i + 1

        while j < len(initial_chunks):
            next_chunk = initial_chunks[j]
            curr_concept = curr_chunk["concept"]
            next_concept = next_chunk["concept"]

            if re.search(level_pattern, curr_concept, re.IGNORECASE) or re.search(level_pattern, next_concept, re.IGNORECASE):
                break

            is_cont = False
            clean_curr = re.sub(r'[^a-zA-Z0-9]', '', re.sub(r'\b(cont\.|continued|cont|part)\b', '', curr_concept, flags=re.IGNORECASE)).lower()
            clean_next = re.sub(r'[^a-zA-Z0-9]', '', re.sub(r'\b(cont\.|continued|cont|part)\b', '', next_concept, flags=re.IGNORECASE)).lower()

            if clean_curr and clean_next and (clean_curr == clean_next or clean_curr in clean_next):
                is_cont = True

            try:
                emb1 = emb_model.encode([curr_concept], normalize_embeddings=True)
                emb2 = emb_model.encode([next_concept], normalize_embeddings=True)
                sim = cosine_similarity(emb1, emb2)[0][0]

                if is_cont or sim >= 0.94:
                    logger.info(f"Merging title '{curr_concept}' with '{next_concept}' (sim={sim:.2f}, is_cont={is_cont})")
                    curr_chunk["content"].extend(next_chunk["content"])
                    j += 1
                else:
                    break
            except Exception as e:
                logger.warning(f"⚠️ Title similarity failed: {e}")
                break

        final_chunks.append(curr_chunk)
        i = j

    # Prepend intro chunks
    final_chunks = intro_chunks + final_chunks
    logger.info(f"Final chunk count after merging: {len(final_chunks)}")

    for idx, ch in enumerate(final_chunks):
        logger.info(f"   Chunk {idx+1}: '{ch['concept']}' (Pages: {ch['page_start']}+)")

    return final_chunks


def chunk_to_embedding_text(chunks, max_chars=1800):
   """
    Converts chunks into clean, embedding-ready text.
    Splits overly large chunks into page-aligned subchunks to maintain context.
    """
   embeddings = []
   for chunk_idx, chunk in enumerate(chunks):
        concept = chunk.get("concept", "").strip()
        units = chunk.get("content", [])
        if not units:
            logger.warning(f"⚠️ Chunk {chunk_idx+1} ('{concept}') has no units. Skipping.")
            continue
        logger.info(f"\n Processing Chunk {chunk_idx+1}: '{concept}'")

        #  Clean units

        effective_units = []
        for u in units:
            u_text = u.get("text", "").strip()
            if not u_text:
                continue
            if u.get("type") in ("image", "chart") and "[Image recovery failed]" in u_text:
                continue
            if "display_formula" in u.get("type", "") or "$" in u_text:
                u_text = latex_to_unicode(u_text)
            elif u.get("type") == "algorithm":
                u_text = normalize_code(u_text)
            u_text = html.unescape(u_text)
            effective_units.append({
                **u,
                "effective_text": u_text
            })

        if not effective_units:
            logger.warning(f"⚠️ Chunk {chunk_idx+1} has no valid units after cleaning.")
            continue

        full_text = "\n\n".join(u["effective_text"] for u in effective_units)

        pages = sorted(set(u["page"] for u in effective_units))
        logger.info(f"   Pages: {pages} | Length: {len(full_text)} chars")

        # If chunk small : single embedding
        if len(full_text) <= max_chars:
            embedding_input = f"Concept: {concept}\nRole: section\n\n{full_text}"
            embeddings.append({
                "embedding_text": embedding_input,
                "metadata": {
                    "concept": concept,
                    "page": effective_units[0].get("page"),
                    "units": effective_units,
                    "total_subchunks": 1,
                }
            })

            logger.info("   ✅ Single subchunk")


        else:
            logger.info(" Splitting by pages...")
            page_to_units = {}
            for u in effective_units:
                page_to_units.setdefault(u["page"], []).append(u)
            subchunks = []
            current_text = ""
            current_pages = []
            for page in sorted(page_to_units.keys()):
                page_units = page_to_units[page]
                page_text = "\n\n".join(u["effective_text"] for u in page_units)
                if current_text and len(current_text) + len(page_text) + 2 > max_chars:
                    subchunks.append({
                        "text": current_text.strip(),
                        "units": [u for p in current_pages for u in page_to_units[p]]
                    })
                    logger.info(f" Subchunk {len(subchunks)} pages {current_pages}")
                    current_text = page_text
                    current_pages = [page]
                else:
                    current_pages.append(page)
                    current_text += ("\n\n" + page_text) if current_text else page_text
            if current_text:
                subchunks.append({
                    "text": current_text.strip(),
                    "units": [u for p in current_pages for u in page_to_units[p]]
                })
                logger.info(f"Subchunk {len(subchunks)} pages {current_pages}")
            logger.info(f"   ✅ Created {len(subchunks)} subchunks")
            # Create embedding entries
            for idx, sc in enumerate(subchunks):
                embedding_input = f"Concept: {concept}\nRole: section\n\n{sc['text']}"
                embeddings.append({
                    "embedding_text": embedding_input,
                    "metadata": {
                        "concept": concept,
                        "page": sc["units"][0].get("page"),
                        "units": sc["units"],
                        "total_subchunks": len(subchunks),
                    }
                })

   logger.info(f"\n Total embeddings generated: {len(embeddings)}")
   return embeddings
logger.info("✅ Step 9 Complete: Chunking logic improved for slides.")

In [ ]:
# @title ⚙️ Step 10: Main PDF Processing Loop (Memory Safe)
# @markdown Runs OCR on every page. Optimized to prevent GPU crashes.

def extract_pdf_info(pdf_name,pdf_path, pipeline):
    """
    Main Loop: PDF -> Images -> OCR -> JSON
    """

    # Use the global directories we set in Step 3
    pdf_image_dir = IMAGE_DIR / pdf_name
    pdf_json_dir = JSON_DIR / pdf_name

    for d in [pdf_image_dir, pdf_json_dir]:
        d.mkdir(parents=True, exist_ok=True)

    final_json_path = pdf_json_dir / f"{pdf_name}.json"

    # Get page count
    try:
        num_pages = pdfinfo_from_path(str(pdf_path))['Pages']
    except:
        logger.warning("⚠️ Could not read PDF info. Installing poppler again...")
        os.system("apt-get install -y poppler-utils")
        num_pages = pdfinfo_from_path(str(pdf_path))['Pages']

    pdf_result = {
        "file_name": pdf_path.name,
        "num_pages": num_pages,
        "pages": []
    }

    logger.info(f"[*] Starting OCR for {pdf_name} ({num_pages} pages)...")
    pbar = tqdm(range(1, num_pages + 1), desc="OCR Processing", unit="page")

    for i in pbar:
        try:
            # 1. CLEAN MEMORY BEFORE STARTING (Critical for Colab)
            paddle.device.cuda.empty_cache()
            gc.collect()

            # 2. Convert PDF Page to Image (LOWER RES to save RAM)
            # Changed dpi=150 -> dpi=100 to prevent OOM
            page = convert_from_path(str(pdf_path), dpi=100, first_page=i, last_page=i)
            page_img = page[0]

            if page_img.width == 0 or page_img.height == 0: continue

            page_np = np.array(page_img).astype("uint8")
            image_path = pdf_image_dir / f"page_{i:03d}.png"
            page_img.save(image_path)

            # 3. Run PaddleOCR
            output = pipeline.predict(page_np)
            result = output[0]

            # 4. Parse Results
            parsing_res_list = [to_dict(block) for block in result.get("parsing_res_list", [])]
            layout_det_res = to_dict(result.get("layout_det_res", {}))

            pbar.set_postfix_str(f"Found {len(parsing_res_list)} blocks", refresh=True)

            pdf_result["pages"].append({
                "page_number": i,
                "image_path": str(image_path),
                "parsing_res_list": parsing_res_list,
                "layout_det_res": layout_det_res,
                "height": page_img.height,
                "width": page_img.width
            })
            ram_usage = psutil.virtual_memory().percent
            logger.info(f"📊 Page {i} - System RAM Usage: {ram_usage}%")

            # Cleanup RAM explicitly
            del page_img, page_np, page, output, result
            del parsing_res_list, layout_det_res
            paddle.device.cuda.empty_cache()
            gc.collect()

        except Exception as e:
            logger.warning(f"⚠️ Error on page {i}: {e}")
            # If a page fails, we try to skip it to save the rest of the document
            continue

    # Save Final JSON
    with open(final_json_path, "w", encoding="utf-8") as f:
        json.dump(pdf_result, f, ensure_ascii=False, indent=2)

    logger.info(f"✅ OCR Complete. Data saved to: {final_json_path}")
    return pdf_result

logger.info("✅ Step 10 Complete: Extraction loop optimized.")

In [ ]:
# @title 🧠 Step 11: Embedding Model Helpers
# @markdown Loads the BGE-M3 model to convert text chunks into vectors.

def load_embedding_model():
    """Loads the SentenceTransformer model on GPU."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    logger.info(f"[*] Loading Embedding Model on {device}...")
    return SentenceTransformer("BAAI/bge-m3", device=device)

def get_embedding(text: str, embModel) -> List[float]:
    """Generates a normalized vector for a given text string."""
    # normalize_embeddings=True improves cosine similarity accuracy
    emb = embModel.encode([text], normalize_embeddings=True)
    return emb[0].tolist()

logger.info("✅ Step 11 Complete: Embedding functions defined.")

In [ ]:
# @title ⚙️ Step 12: Extraction Pipeline

def run_friend_extraction_pipeline(pdf_name,pdf_path,pipeline_manager):
    """
    Runs the full OCR -> Chunking -> Embedding pipeline on a single PDF.
    Refactored from 'main()' to be modular.
    """
    logger.info(f"Starting Extraction Pipeline for: {pdf_path.name}")
    logger.info(f"📊 [PIPELINE START] RAM Usage: {psutil.virtual_memory().percent}%")

    pdf_name_raw = pdf_path.stem

    # 1. OCR & Layout Analysis
    VL_Pipeline = pipeline_manager.get_vl_pipeline()

    ocr_result = extract_pdf_info(pdf_name,pdf_path, VL_Pipeline)
    logger.info(f"✅ OCR Complete. 📊 RAM: {psutil.virtual_memory().percent}%")

    # 2. Convert to Units
    units = blocks_to_units(ocr_result, pipeline_manager)
    del ocr_result
    gc.collect()

    # **Explicitly unload vision models to free up GPU memory**
    pipeline_manager.unload_all()

    # Use sanitized name for unified_json directory
    unified_dir = UNIFIED_DIR / pdf_name
    unified_dir.mkdir(parents=True, exist_ok=True)

    # Use sanitized name for units file
    units_path = unified_dir / f"{pdf_name}_units.json"
    with open(units_path, "w", encoding="utf-8") as f:
        json.dump(units, f, ensure_ascii=False, indent=2)
    logger.info(f"✅ Created {len(units)} raw units. 📊 RAM: {psutil.virtual_memory().percent}%")

    # 3. Semantic Chunking
    logger.info("[*] Loading Embedding Model...")
    embedding_model = load_embedding_model()

    chunks = process_units_to_chunks(units, embedding_model)
    del units
    gc.collect()
    # Use sanitized name for chunks file
    chunks_path = unified_dir / f"{pdf_name}_chunks.json"
    with open(chunks_path, "w", encoding="utf-8") as f:
        json.dump({"doc_id": pdf_name, "chunks": chunks}, f, ensure_ascii=False, indent=2)
    logger.info(f"✅ Generated {len(chunks)} chunks. 📊 RAM: {psutil.virtual_memory().percent}%")

    # 4. Prepare Embeddings
    # Use sanitized name for embedding_inputs directory
    embedding_dir = EMBEDDING_DIR / pdf_name
    embedding_dir.mkdir(parents=True, exist_ok=True)
    # Use sanitized name for embedding input texts file
    embedding_path = embedding_dir / f"{pdf_name}_embedding_input_texts.json"
    embedding_inputs = []

    for idx, item in enumerate(chunk_to_embedding_text(chunks)):
        text = item["embedding_text"]
        metadata = item["metadata"]
        if not text.strip(): continue

        embedding_inputs.append({
            "embedding_id": f"{pdf_name}_c{idx:04d}", # Use raw name for embedding ID if needed for lookup back to original doc
            "page": metadata.get("page"),
            "text": text,
            "metadata": metadata
        })

    del chunks
    gc.collect()

    with open(embedding_path, "w", encoding="utf-8") as f:
          json.dump({"doc_id": pdf_name, "inputs": embedding_inputs}, f, ensure_ascii=False, indent=2)

    logger.info(f"✅ Embeddings Prepared. 📊 RAM: {psutil.virtual_memory().percent}%")

    # 5. Store in ChromaDB (Using Global DB_DIR)
    chroma_client = chromadb.PersistentClient(path=str(DB_DIR))
    collection_name = f"cs_podcast_{pdf_name}"

    try:
        chroma_client.delete_collection(name=collection_name)
    except: pass

    collection = chroma_client.get_or_create_collection(
        name=collection_name,
        metadata={"hnsw:space": "cosine"}
    )

    logger.info(f"[*] Storing {len(embedding_inputs)} vectors in DB...")

    ids, embeddings, documents, metadatas = [], [], [], []
    idx = 0
    for item in tqdm(embedding_inputs, desc="Embedding"):
        emb_id = item["embedding_id"]
        text = item["text"]
        meta = item["metadata"]

        emb = get_embedding(text, embedding_model)

        chroma_meta = {
            "doc_id": pdf_name,
            "concept": meta.get("concept", ""),
            "page": meta.get("page", 0),
            "global_subchunk": idx+1,
            "total_subchunks": int(meta.get("total_subchunks", 0)),
            "unit_ids": ",".join([str(u.get("unit_id", "")) for u in meta.get("units", [])])
          }
        idx += 1
        ids.append(emb_id)
        embeddings.append(emb)
        documents.append(text)
        metadatas.append(chroma_meta)

    collection.add(
        ids=ids,
        embeddings=embeddings,
        documents=documents,
        metadatas=metadatas
    )

    # Final Pipeline Cleanup
    del ids, embeddings, documents, metadatas, embedding_inputs

    # STRICT EMBEDDING CLEANUP: Force model off GPU before deleting
    try:
        embedding_model.to('cpu')
    except:
        pass
    del embedding_model

    force_system_ram_cleanup()

    logger.info(f"🎉 Pipeline Success! Data is ready for RAG. 📊 Final RAM: {psutil.virtual_memory().percent}%")
logger.info("✅ Step 12 Complete: extraction pipeline packaged as a function.")


In [ ]:
# @title 🔍 Step 13: Retrieval System
class Doc2PodRetrieval:
    def __init__(self, db_path, collection_name, use_gpu=True, debug=False):
        self.device = "cuda" if use_gpu else "cpu"
        self.debug = debug
        self.client = chromadb.PersistentClient(path=str(db_path))
        self.collection = self.client.get_collection(name=collection_name)
        self.emb_model = SentenceTransformer("BAAI/bge-m3", device=self.device)

        if self.debug:
            logger.info(f"✅ Loaded collection '{collection_name}' with {self.collection.count()} vectors")

    def unload(self):
        try:
            self.emb_model.to('cpu')
        except:
            pass
        del self.emb_model
        force_system_ram_cleanup()

    def _format_results(self, subchunks):
        if not subchunks:
          return [],[]

        unique_concepts = []
        seen = set()
        for sc in subchunks:
            c = sc["metadata"]["concept"]
            if c not in seen:
                unique_concepts.append(c)
                seen.add(c)

        results = []
        for sc in subchunks:
            meta = sc["metadata"]
            next_concept = None
            if meta["concept"] in unique_concepts:
                idx = unique_concepts.index(meta["concept"])
                if idx + 1 < len(unique_concepts):
                    next_concept = unique_concepts[idx + 1]

            results.append({
                "text": sc["text"],
                "concept": meta["concept"],
                "total_subs": meta.get("total_subchunks", 0),
                "next_concept": next_concept
            })
        return results,unique_concepts

    def get_context(self, query=None, page_range=None, k=10):
        """
        Unified retrieval that works with SUBCHUNKS.
        Returns list of {text, concept, total_subs, next_concept} dicts.
        """
        target_data = []
        where_filter = None

        if page_range:
            start_page, end_page = page_range
            where_filter = {
                "$and": [
                    {"page": {"$gte": start_page}},
                    {"page": {"$lte": end_page}}
                ]
            }
        if not query:
            results = self.collection.get(
                where=where_filter,
                limit=self.collection.count() if self.collection.count() > 0 else 100,
                include=["documents", "metadatas"]
            )
            for text, meta in zip(results['documents'], results['metadatas']):
                target_data.append({"text": text, "metadata": meta})
        else:
          query_emb = self.emb_model.encode([query], normalize_embeddings=True)[0].tolist()
          chroma_results = self.collection.query(
              query_embeddings=[query_emb],
              n_results=k,
              where=where_filter,
             include=["documents", "metadatas", "distances"]
          )
          for text, meta in zip(chroma_results['documents'][0], chroma_results['metadatas'][0]):
              target_data.append({"text": text, "metadata": meta})
        target_data.sort(key=lambda x: int(x["metadata"].get("global_subchunk", 0)))
        if self.debug:
            logger.info(f"Retrieved {len(target_data)} chunks from DB.")

        return self._format_results(target_data)

In [ ]:
class ScriptWriter:
    def __init__(self):
        self.last_summary = ""
        self.concept_subcounter = {}

    def reset(self):
        self.last_summary = ""
        self.concept_subcounter = {}

    def build_generation_prompt(self, context, current_topic, doc_structure, sub_idx, total_subs, next_concept, is_first, current_sec, total_sec, prev_topic):

        position_parts = []
        continuity_parts = []

        if current_sec and total_sec:
          position_parts.append(
              f"Based on the doc_structure:\n{doc_structure}.\n"
              f"You are currently explaining section {current_sec}.\n"
              f"which corresponds to the title '{current_topic}'.\n"
              )
        if current_sec > 1:
          position_parts.append(
            "You MUST behave as if all previous sections are fully explained.\n"
            )

        if total_subs > 1:
            position_parts.append(
                f"This section is divided into {total_subs} sub-parts.\n"
                f"You are now at sub-part {sub_idx}.\n"
                )
            if sub_idx != 1:
                position_parts.append("Do NOT treat this as a new concept.\n")

        if prev_topic:
            position_parts.append(f"Transition naturally from '{prev_topic}'.")

        if self.last_summary:
            position_parts.append(
                "You MUST continue from the previous summary and the content already explained.\n"
                "Do NOT repeat any previously explained content. "
                "Mention key ideas AND examples already used. "
                f"Context from previous parts:\n {self.last_summary}.\n")

        if is_first and sub_idx == 1:
            continuity_parts.append(
                "You MUST start with a warm Egyptian podcast-style introduction.\n"
                'Mention the podcast name "كود وكلام" naturally once.\n'
                "Include a casual greeting between Sara and Ahmed.\n"
                "Do NOT start explaining immediately.\n"
                "Do NOT mention or reference any previous episode.\n"
                )

        if total_subs > 1:
            if sub_idx == 1:
              if is_first:
                continuity_parts.append(
                    f"Continue with a short hook question in Egyptian Arabic related to '{current_topic}'.\n"
                    "Then continue naturally. "
                    )
              else:
                continuity_parts.append(
                    f"Start with a short hook question in Egyptian Arabic related to '{current_topic}'.\n"
                    "Then continue naturally. "
                    )
              continuity_parts.append(
                f"Ahmed MUST focus ONLY on explaining the current point: '{current_topic}'.\n"
                "Ahmed's final sentence must be a solid conclusion to this specific point.\n"
                "STRICTLY FORBIDDEN: Do not mention, hint at, or transition to any upcoming topics or sub-parts.\n"
                "End the script abruptly after Ahmed's last explanation sentence (No closing remarks, no 'wait for part 2').\n"
            )

            elif sub_idx < total_subs:
                continuity_parts.append(
                f"You MUST continue naturally from the previous sub-part without any greeting.\n"
                "Sara MUST start by briefly acknowledging the previous point, then bridge to the rest of the topic.\n"
                f"Do NOT repeat explanations of '{current_topic}'.\n"
                "Start with a short linking sentence in Egyptian Arabic.\n"
            )

            else:
                continuity_parts.append(
                "You MUST continue naturally from the previous sub-part without any greeting.\n"
                "After finishing the explanation, You MUST conclude this final sub-part with a short, clear conclusion in Egyptian Arabic.\n"
                "Use both the previous summary and the current sub-part content to produce one cohesive final wrap-up of the entire concept.\n"
                )
                if not next_concept:
                    continuity_parts.append(
                        "End with a professional thank-you exchange..\n"
                        "Do not mention future episodes.\n"
                    )
                else:
                    continuity_parts.append(
                        "Then smoothly transition to the next topic.\n"
                        "End immediately after Ahmed's final explanation sentence.\n"
                        "Do not include any closing remarks, thanks, goodbyes, or additional dialogue after this sentence.\n"
                    )
        else:
          continuity_parts.append(
              f"Start with a short hook question in Egyptian Arabic related to '{current_topic}'.\n"
              "You MUST conclude this sub-part clearly. "
              "Do NOT make an abrupt transition. Continue in Egyptian Arabic.\n"
              )
          if current_sec >= 1:
            continuity_parts.append("Do NOT re-explain concepts already introduced.\n")

          if not next_concept:
              continuity_parts.append(
                "End with a professional thank-you exchange..\n"
                "Do not mention future episodes.\n"
            )
          else:
              continuity_parts.append(
              "Then smoothly transition to the next topic.\n"
              "End immediately after Ahmed's final explanation sentence.\n"
              "Do not include any closing remarks, thanks, goodbyes, or additional dialogue after this sentence.\n"
              )
        continuity_rules = "\n".join(continuity_parts)
        position_info = "\n".join(position_parts)

        user_prompt = f"""
        أنت كاتب سيناريو محترف لبرنامج "بودكاست تقني" باللهجة المصرية القاهرية.
        هدفنا: شرح مفاهيم علوم الحاسب باستخدام تشبيهات شعبية.

        ### الشخصيات:
        - Speaker 1 (سارة): المذيعة. لسانها مصري جداً، دمها خفيف، بتسأل بذكاء وتقاطع أحمد كل شوية عشان الجمهور يفهم.
        - Speaker 2 (أحمد): الضيف (Senior Engineer). خبير، صوته هادي، ممنوع يشرح شرح أكاديمي.
          لازم يشرح بأمثلة شعبية (مطبخ، سوق، زحمة، مواصلات).

        Alternate strictly: Sara speaks, then Ahmed replies, repeat. No speaker twice in a row.

        ### Position and structure
        {position_info}

        ### CONTINUITY RULES
        {continuity_rules}


        ### FOCUS CONTENT (Technical text to convert to dialogue):
        {context}

        ### START DIALOGUE:
        سارة:
        """
        return user_prompt.strip()

    def get_summary_prompt(self, concept, text_preview):
      if self.last_summary:
        task_instruction = f"Update the following previous summary by incorporating the new content: {self.last_summary}"
      else:
        task_instruction = "Create a concise initial summary based on the new content."
      user_prompt = f"""
      You are an expert content summarizer focusing on the concept: "{concept}"
      New content: {text_preview}
      Task: {task_instruction}
      Requirements:
      - Keep it short: 2-4 sentences max.
      - Summarize only the progress made on this concept so far.
      - Clear, concise, and English only.
      """.strip()
      return user_prompt

In [ ]:
# @title 🎙️ Step 15: Podcast Studio

class PodcastStudio:
    def __init__(self, pipeline_manager):
        self.manager = pipeline_manager
        self.writer = ScriptWriter()



    def _call_llm(self, user_prompt):

        gc.collect()
        torch.cuda.empty_cache()

        model, tokenizer = self.manager.get_qwen3_instruct_model()
        messages = [
            {"role": "user", "content": user_prompt}
        ]

        text_prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text_prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=1500,
                temperature=0.8,
                top_p=0.9,
                repetition_penalty=1.2,
                pad_token_id=tokenizer.eos_token_id
            )

        input_length = inputs["input_ids"].shape[1]
        result = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
        # Clean Chinese/Asian characters automatically
        result = re.sub(r'[\u4e00-\u9fff]+', '', result)

        del inputs, outputs, messages, text_prompt
        gc.collect()
        torch.cuda.empty_cache()

        return result

    def generate_podcast(self, subchunks, concepts_list):
        self.writer.reset()
        script_segments = []
        outputs = []

        actual_totals = {}
        for s in subchunks:
          c = s.get("concept", "Unknown")
          actual_totals[c] = actual_totals.get(c, 0) + 1

        if concepts_list:
            structure = "This lecture covers these topics in order:\n" + "\n".join(
                [f"{i+1}. {c}" for i, c in enumerate(concepts_list)]
            )
        else:
            structure = "Structure unavailable."

        prev_concept = None
        logger.info(f"Chunks count: ")
        logger.info(f"📊 [STUDIO START] RAM Usage: {psutil.virtual_memory().percent}%")


        for idx, sub in enumerate(subchunks):
            logger.info(f"{idx}: {len(sub)} chars")
            curr_concept = sub["concept"]
            total_subs = actual_totals.get(curr_concept, 1)

            if curr_concept != prev_concept:
              self.writer.last_summary = ""

            self.writer.concept_subcounter[curr_concept] = self.writer.concept_subcounter.get(curr_concept, 0) + 1
            current_sub_idx = self.writer.concept_subcounter[curr_concept]

            next_c = None
            if curr_concept in concepts_list:
                c_idx = concepts_list.index(curr_concept)
                if c_idx + 1 < len(concepts_list):
                    next_c = concepts_list[c_idx + 1]

            user_prompt= self.writer.build_generation_prompt(
                context=sub["text"],
                current_topic=curr_concept,
                doc_structure=structure,
                sub_idx=current_sub_idx,
                total_subs=total_subs,
                next_concept=next_c,
                is_first=(idx == 0),
                current_sec=concepts_list.index(curr_concept) + 1,
                total_sec=len(concepts_list),
                prev_topic=(prev_concept if curr_concept != prev_concept else None)
            )

            logger.info(f"[*] Studio: Generating {curr_concept} ({current_sub_idx}/{total_subs})...")
            logger.info(f"📊 Before LLM Call RAM: {psutil.virtual_memory().percent}%")
            script_part = self._call_llm(user_prompt)
            script_segments.append(script_part)
            logger.info(f"📊 After LLM Script RAM: {psutil.virtual_memory().percent}%")

            u_summary_p= self.writer.get_summary_prompt(curr_concept, sub["text"])
            self.writer.last_summary = self._call_llm(u_summary_p)
            prev_concept = curr_concept

            del user_prompt, u_summary_p
            sub["text"] = ""

            force_system_ram_cleanup()
            logger.info(f"📊 After Loop Cleanup RAM: {psutil.virtual_memory().percent}%")

        return script_segments

In [ ]:
# @title 🌐 Step 16: Doc2Pod API Server (Thread-Safe)
# @markdown Defines the FastAPI application, background task orchestration, and Cloud storage integration.

import re
import uuid
import glob
import subprocess
import threading
import traceback
import numpy as np
import soundfile as sf
import requests
from fastapi import FastAPI, HTTPException, status
from pydantic import BaseModel

# ==========================================
# API UTILITIES
# ==========================================

def sanitize_filename(text):
    if not text:
        return "default_collection"
    safe = re.sub(r'[^a-zA-Z0-9]', '_', text.strip())
    safe = safe[:30]
    safe = safe.strip('_')
    return safe

def get_pdf_selection():
    pdf_files = list(PDF_DIR.glob("*.pdf"))
    if not pdf_files:
        logger.error(f"\n❌ No PDFs found in {PDF_DIR}!")
        return None

    logger.info("\nAvailable Documents:\n")
    for idx, f in enumerate(pdf_files):
        logger.info(f"{idx+1}. {f.name}")

    while True:
        choice = input(f"\nSelect Document (1-{len(pdf_files)}): ")
        if choice.isdigit() and 1 <= int(choice) <= len(pdf_files):
            return pdf_files[int(choice)-1]

def normalize_script(script: str):
    """Formats the script cleanly, strictly enforcing Speaker 1 (Sara) and Speaker 2 (Ahmed)."""
    lines = script.split("\n")
    normalized = []

    for line in lines:
        line = line.strip()
        if not line:
            continue

        lower_line = line.lower()

        if re.match(r'^(سارة|sara|speaker 1).*?:', lower_line):
            text = re.sub(r'^(سارة|sara|speaker 1).*?:', '', line, flags=re.IGNORECASE).strip()
            normalized.append(f"Speaker 1: {text}")

        elif re.match(r'^(أحمد|احمد|ahmed|speaker 2).*?:', lower_line):
            text = re.sub(r'^(أحمد|احمد|ahmed|speaker 2).*?:', '', line, flags=re.IGNORECASE).strip()
            normalized.append(f"Speaker 2: {text}")

        else:
            if len(normalized) > 0:
                normalized[-1] += f" {line}"

    return "\n".join(normalized)


# ==========================================
# SERVER INITIALIZATION & CONFIG
# ==========================================
SUPABASE_URL = userdata.get("SUPABASE_URL")
SUPABASE_KEY = userdata.get("SUPABASE_KEY")

PDF_BUCKET = "PDFs"
SCRIPT_BUCKET = "Scripts"
AUDIO_BUCKET = "Podcasts"

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)
# OUTPUT_DIR = BASE_DIR / "output"
# OUTPUT_DIR.mkdir(exist_ok=True)

app = FastAPI(title="Doc2Pod Studio API", version="1.0")

tasks_db = {}
pipeline_manager = LazyPipelineManager()
studio = PodcastStudio(pipeline_manager)

# GLOBAL LOCK: Prevents multiple heavy tasks from running simultaneously and crashing VRAM
is_processing = False
processing_lock = threading.Lock()

class GenerateRequest(BaseModel):
    file_key: str
    mode: int
    topic: str | None = None
    start_page: int | None = None
    end_page: int | None = None

# ==========================================
# CORE ENDPOINTS
# ==========================================
@app.post("/generate")
async def generate_podcast(request: GenerateRequest):
    global is_processing

    # 1. Concurrency Check (Reject if busy)
    with processing_lock:
        if is_processing:
            logger.warning("⚠️ Request rejected: Server is currently busy.")
            raise HTTPException(
                status_code=status.HTTP_503_SERVICE_UNAVAILABLE,
                detail="The AI Server is currently busy generating another podcast. Please wait."
            )
        is_processing = True

    logger.info(f"📥 INCOMING REQUEST: Mode {request.mode} | File: {request.file_key}")

    try:
        # 2. Download Target PDF from Supabase
        signed_res = supabase.storage.from_(PDF_BUCKET).create_signed_url(request.file_key, 300)
        if not signed_res or "signedURL" not in signed_res:
            raise Exception("Failed to generate secure download URL from Supabase.")

        download_url = signed_res["signedURL"]
        response = requests.get(download_url)
        if response.status_code != 200:
            raise Exception("Failed to download the document.")

        file_bytes = response.content
        task_id = str(uuid.uuid4())
        tasks_db[task_id] = {"status": "PROCESSING"}

        # 3. Background Orchestrator
        def heavy_processing_task():
            global is_processing
            try:
                # --- Save Local Copy ---
                pdf_path = PDF_DIR / f"{request.file_key}"
                with open(pdf_path, "wb") as f:
                    f.write(file_bytes)

                safe_name = sanitize_filename(pdf_path.stem)
                coll_name = f"cs_podcast_{safe_name}"
                chunks_json_path = UNIFIED_DIR / safe_name / f"{safe_name}_chunks.json"

                # --- Execute Extraction Pipeline ---
                if not chunks_json_path.exists():
                    run_friend_extraction_pipeline(safe_name, pdf_path, pipeline_manager)
                    if not chunks_json_path.exists():
                        raise Exception("Data extraction pipeline failed.")
                else:
                   logger.info("Skipping Extraction Pipeline Data already exists")

                pipeline_manager.unload_all()

                # --- RAG based on Mode ---
                rag = Doc2PodRetrieval(DB_DIR, coll_name)

                if request.mode == 3:   # Full Document
                    target_subchunks, current_concepts = rag.get_context()
                elif request.mode == 2: # Topic + Page Range
                    target_subchunks, current_concepts = rag.get_context(
                        query=request.topic, page_range=(request.start_page, request.end_page)
                    )
                elif request.mode == 1: # Topic Only
                    target_subchunks, current_concepts = rag.get_context(query=request.topic)
                else:
                    raise Exception("Invalid Generation Mode.")

                if not target_subchunks:
                    raise Exception("No relevant content found for the given parameters.")

                # Clean up RAG memory properly
                rag.unload()
                del rag
                force_system_ram_cleanup()

                # --- Script Generation (LLM) ---
                script_segments = studio.generate_podcast(target_subchunks, current_concepts)
                final_script = "\n\n".join(script_segments)
                pipeline_manager.unload_all()

                # Determine correct output directory based on mode
                if request.mode == 1: out_dir = MODE1_DIR
                elif request.mode == 2: out_dir = MODE2_DIR
                else: out_dir = MODE3_DIR

                # Use task_id for unique folder creation
                task_dir = out_dir / task_id
                task_dir.mkdir(exist_ok=True)

                # Save RAW script (without normalization)
                raw_script_name = f"{task_id}_mode{request.mode}_raw.txt"
                raw_txt_path = task_dir / raw_script_name
                with open(raw_txt_path, "w", encoding="utf-8") as f:
                    f.write(final_script)

                # Save NORMALIZED script
                script_name = f"{task_id}_mode{request.mode}_normalized.txt"
                txt_path = task_dir / script_name
                normalized = normalize_script(final_script)
                with open(txt_path, "w", encoding="utf-8") as f:
                    f.write(normalized)

                # ---  Audio Generation (TTS) ---
                logger.info("🎙️ Initializing TTS Engine...")
                pipeline_manager.ensure_voice_model_loaded()

                lines = [line.strip() for line in normalized.strip().split('\n') if line.strip()]
                chunk_size = 2
                chunks = [lines[i:i + chunk_size] for i in range(0, len(lines), chunk_size)]

                logger.info(f"Audio Strategy: {len(lines)} lines divided into {len(chunks)} inference chunks.")

                generated_files = []
                for i, chunk in enumerate(chunks):
                    chunk_text = "\n".join(chunk)
                    chunk_txt_path = task_dir / f"chunk_{i}.txt"
                    chunk_out_path = task_dir / f"chunk_audio_{i}"
                    chunk_out_path.mkdir(exist_ok=True)

                    with open(chunk_txt_path, "w", encoding="utf-8") as f:
                        f.write(chunk_text)

                    logger.info(f"--- ⏳ Synthesizing Audio Chunk {i+1}/{len(chunks)} ---")
                    force_system_ram_cleanup()

                    cmd = [
                        "python", "/content/VibeVoice/demo/inference_from_file.py",
                        "--model_path", "/content/egyptian-model",
                        "--txt_path", str(chunk_txt_path),
                        "--speaker_names", "egyptian_female", "egyptian_male",
                        "--output_dir", str(chunk_out_path),
                        "--cfg_scale", "3.3",
                        "--seed", "0"
                    ]

                    subprocess.run(cmd, capture_output=True, text=True)

                    wavs = glob.glob(f"{chunk_out_path}/*.wav")
                    if wavs:
                        latest_wav = max(wavs, key=os.path.getctime)
                        generated_files.append(latest_wav)
                        logger.info(f"✅ Chunk {i+1} synthesized successfully.")
                    else:
                        logger.warning(f"❌ Chunk {i+1} failed to synthesize.")

                    del chunk_text, cmd, wavs
                    force_system_ram_cleanup()

                if not generated_files:
                     raise Exception("Complete failure in audio synthesis. No chunks generated.")

                # --- Audio Merging & Compression ---
                logger.info("🔄 Compiling final audio track...")
                audio_data = []
                sample_rate = None
                silence_duration = 0.5

                for wav_file in generated_files:
                    data, sr = sf.read(wav_file)
                    if sample_rate is None:
                        sample_rate = sr
                        silence = np.zeros(int(sample_rate * silence_duration))

                    audio_data.append(data)
                    audio_data.append(silence)

                if audio_data:
                    audio_data = audio_data[:-1]

                final_audio = np.concatenate(audio_data)
                temp_wav_path = task_dir / f"temp_{task_id}.wav"
                sf.write(str(temp_wav_path), final_audio, sample_rate)

                audio_name = f"{task_id}_mode{request.mode}.mp3"
                mp3_path = task_dir / audio_name

                # Compress to MP3
                subprocess.run([
                    "ffmpeg", "-y", "-i", str(temp_wav_path), "-vn",
                    "-ar", "44100", "-ac", "1", "-b:a", "64k", str(mp3_path)
                ], capture_output=True)

                mb = os.path.getsize(str(mp3_path)) / 1024**2
                logger.info(f"🎉 Final MP3 compiled successfully! ({mb:.1f} MB)")

                # --- Cloud Upload ---
                with open(txt_path, "rb") as f:
                    supabase.storage.from_(SCRIPT_BUCKET).upload(
                        path=script_name, file=f,
                        file_options={"content-type": "text/plain", "upsert": "true"}
                    )

                with open(mp3_path, "rb") as f:
                    supabase.storage.from_(AUDIO_BUCKET).upload(
                        path=audio_name, file=f,
                        file_options={"content-type": "audio/mpeg", "upsert": "true"}
                    )

                logger.info("☁️ Assets synced to Supabase.")

                tasks_db[task_id] = {
                    "status": "DONE",
                    "script": {"file_key": script_name},
                    "audio": {"file_key": audio_name}
                }

            except Exception as e:
                logger.error(f"❌ PIPELINE ERROR: {str(e)}")
                logger.error(traceback.format_exc())
                tasks_db[task_id] = {"status": "ERROR", "error": str(e)}

            finally:
                # Ensures absolute cleanup whether success or failure before unlocking.
                logger.info("🧹 Performing aggressive final memory sweep...")
                try:
                    pipeline_manager.unload_all()
                    force_system_ram_cleanup()
                except Exception as cleanup_e:
                    logger.warning(f"Memory cleanup warning: {cleanup_e}")

                # Release the lock for the next user
                with processing_lock:
                    is_processing = False
                logger.info("🔓 Server lock released. Ready for new requests.")

        # Dispatch background thread
        threading.Thread(target=heavy_processing_task, daemon=True).start()
        return {"task_id": task_id}

    except Exception as ex:
        with processing_lock:
            is_processing = False
        raise HTTPException(status_code=500, detail=str(ex))

@app.get("/status/{task_id}")
def get_task_status(task_id: str):
    """Polling endpoint for the Backend to check task completion."""
    task = tasks_db.get(task_id)
    if not task:
        return {"status": "NOT_FOUND"}

    if task["status"] != "DONE":
        return task

    return {
        "status": "DONE",
        "script_path": task["script"]["file_key"],
        "audio_path": task["audio"]["file_key"]
    }

@app.get("/")
def health_check():
    """Simple health check to verify server is active."""
    return {"status": "alive"}

logger.info("✅ Step 16 Complete: API Server successfully configured.")

In [ ]:
# @title 🚀 Step 17: Launch Doc2Pod Server (Cloudflare Tunnel)
# @markdown Executes the server and opens a secure public tunnel for the C# Backend.

import uvicorn
from threading import Thread
import time
import requests
import subprocess
import re
import os
import sys
import nest_asyncio
import asyncio

# Refresh the logger to point to THIS cell's output screen!
logger = setup_system_logger()

# Enable nested event loops for Uvicorn
nest_asyncio.apply()

logger.info("🧹 Cleaning up legacy background processes...")
# Forcefully terminate any existing server or tunnel instances
os.system("pkill -9 uvicorn")
os.system("pkill -9 cloudflared")
os.system("fuser -k 8000/tcp > /dev/null 2>&1")
time.sleep(3) # System cool-down


def keep_server_alive(port=8000, interval=60):
    """Worker thread to ping the server and prevent Colab idle timeout."""
    url = f"http://127.0.0.1:{port}/"
    while True:
        try:
            requests.get(url, timeout=5)
        except:
            pass
        time.sleep(interval)

def start_cloudflare():
    """Starts Cloudflare Tunnel and extracts the public URL with timeout handling."""

    logger.info("☁️ Initializing Cloudflare Tunnel...")

    process = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

    start_time = time.time()
    found = False
    logger.info("⏳ Waiting for Cloudflare to generate link (Timeout: 30s)...")

    # Read output stream to capture the dynamic .trycloudflare.com link
    while True:
        line = process.stdout.readline()
        if not line:
            break

        # Search for the link in the output stream
        match = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)
        if match:
            public_url = match.group(0)
            print(f"\n✅ " + "="*45)
            print(f"✅ Cloudflare API Link: {public_url}")
            print(f"✅ " + "="*45 + "\n")
            logger.info(f"✅ Cloudflare Link captured: {public_url}")
            found = True
            break

        # Safety timeout after 30 seconds
        if time.time() - start_time > 30:
            break

    if not found:
        logger.error("❌ ERROR: Failed to acquire Cloudflare link within 30 seconds.")
        logger.warning("Try restarting the cell or checking the Colab internet connection.")

# Start the Anti-Idle ping worker
Thread(target=keep_server_alive, daemon=True).start()

# Start the tunnel
start_cloudflare()

logger.info("◐️ Starting Uvicorn Server in FOREGROUND mode...")

# Uvicorn configuration for high-concurrency and long-running tasks
config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=8000,
    log_config=None, # Utilizing our custom unified logger
    timeout_keep_alive=600
)
server = uvicorn.Server(config)

# Execute the server on the existing event loop
loop = asyncio.get_event_loop()
loop.run_until_complete(server.serve())

In [ ]:
# @title 🎧 Play the latest generated podcast
import glob
import os
from IPython.display import Audio, display

search_pattern = str(BASE_DIR / "Mode*" / "*" / "*.mp3")
mp3_files = glob.glob(search_pattern)

if mp3_files:
    latest_mp3 = max(mp3_files, key=os.path.getctime)
    mb_size = os.path.getsize(latest_mp3) / (1024 * 1024)

    print(f"🎉 Podcast found!")
    print(f"📁 Path: {latest_mp3}")
    print(f"⚖️ Size: {mb_size:.2f} MB")

    display(Audio(latest_mp3))
else:
    print("❌ No MP3 files have finished generating yet. Wait for the Logs to confirm the MP3 is ready and try again!")